# PySpark: RDD vs DataFrame

---

## 1. Introducción

Apache Spark es un motor de procesamiento distribuido diseñado para trabajar con grandes volúmenes de datos de forma eficiente.

Dentro de Spark existen distintas abstracciones para manejar datos:

- RDD (Resilient Distributed Dataset) → Bajo nivel (fundacional)
- DataFrame → Alto nivel (estándar actual)
- Dataset → Tipado (principalmente en Scala/Java)

Este documento explica en profundidad:
- Arquitectura
- Funcionamiento interno
- Ejecución
- Optimización
- Casos reales
- Comparación técnica detallada

---

## 2. Arquitectura de Spark

### 2.1 Componentes principales

#### Driver
- Define el programa
- Construye el DAG
- Coordina ejecución

#### Cluster Manager
- Gestiona recursos
- Ejemplos:
  - Standalone
  - YARN
  - Kubernetes

#### Executors
- Ejecutan tareas
- Procesan particiones
- Mantienen datos en memoria

---

### 2.2 Flujo de ejecución

1. Usuario define operaciones  
2. Spark construye DAG  
3. Se optimiza el plan  
4. Se divide en stages  
5. Se ejecuta en paralelo  

---

## 3. RDD (Resilient Distributed Dataset)

### 3.1 Definición

Un RDD es una colección distribuida, inmutable y tolerante a fallos que permite procesamiento paralelo.

---

### 3.2 Características fundamentales

#### Inmutabilidad
- No se modifica
- Cada operación genera un nuevo RDD

#### Lazy Evaluation

```python
rdd2 = rdd.map(lambda x: x * 2)  # No ejecuta
rdd2.collect()  # Ejecuta


# Optimización en Spark: Catalyst Optimizer en Profundidad Avanzada

---

## 1. Introducción

En Apache Spark, el uso de DataFrames y Spark SQL permite describir transformaciones de datos de forma declarativa. Esto habilita a Spark a aplicar optimizaciones automáticas antes de ejecutar cualquier operación.

El componente central de este proceso es el Catalyst Optimizer, encargado de transformar un plan lógico en un plan físico altamente eficiente.

Este documento profundiza en:
- Arquitectura interna del optimizador
- Tipos de reglas
- Estrategias de ejecución
- Costos en sistemas distribuidos
- Detalles internos que afectan performance real

---

## 2. Filosofía del optimizador

Spark separa completamente:

- Qué hacer (definido por el usuario)
- Cómo hacerlo (decidido por Spark)

Esto permite aplicar múltiples optimizaciones sin modificar el código original.

---

## 3. Pipeline completo de optimización

Spark transforma una consulta en múltiples etapas:

1. Parsed Logical Plan  
2. Analyzed Logical Plan  
3. Optimized Logical Plan  
4. Physical Plan  
5. Ejecución  

Cada etapa transforma un árbol de operaciones.

---

## 4. Representación interna: Árboles

Todos los planes en Spark son representados como árboles:

Ejemplo:

```text
Project [area]
  └── Filter (monto > 100)
        └── Relation [area, monto]

In [0]:
# Ejemplo de uso de DataFrame en PySpark

# Crear un DataFrame a partir de una lista de diccionarios
data = [
    {"nombre": "Ana", "edad": 23},
    {"nombre": "Luis", "edad": 31},
    {"nombre": "Marta", "edad": 29}
]
df = spark.createDataFrame(data)

# Mostrar el DataFrame
display(df)

# Filtrar filas donde la edad es mayor a 25
df_filtrado = df.filter(df.edad > 25)
display(df_filtrado)

# Agrupar por nombre y calcular la edad máxima
df_agrupado = df.groupBy("nombre").agg({"edad": "max"})
display(df_agrupado)

# Ejemplo de uso de DataFrame en PySpark (equivalente a lo explicado con RDD)

# Ejemplo de inmutabilidad: cada transformación crea un nuevo DataFrame, el original no cambia
df_filtrado = df.filter(df.edad > 25)  # DataFrame nuevo, df original sigue igual
df_mapeado = df.withColumn("edad_mas_1", df.edad + 1)  # Otro DataFrame nuevo

# Recoger los resultados filtrados
resultados_df = df_filtrado.collect()

# Mostrar los resultados filtrados
for row in resultados_df:
    print(f"Nombre: {row['nombre']}, Edad: {row['edad']}")

# Seleccionar solo los nombres
df_nombres = df.select("nombre")
print([row['nombre'] for row in df_nombres.collect()])

# flatMap equivalente: explode solo array de un solo tipo (ejemplo con 'nombre')
from pyspark.sql.functions import array, explode
df_flat = df.select(explode(array("nombre")).alias("elemento"))
print([row['elemento'] for row in df_flat.collect()])

# distinct: elimina duplicados
data_con_duplicados = [
    {"nombre": "Ana", "edad": 23},
    {"nombre": "Ana", "edad": 23},
    {"nombre": "Luis", "edad": 31}
]
df_con_duplicados = spark.createDataFrame(data_con_duplicados)
df_distinct = df_con_duplicados.distinct()
print(df_distinct.collect())

# union: une dos DataFrames
df_extra = spark.createDataFrame([{"nombre": "Pedro", "edad": 40}])
df_union = df.union(df_extra)
print(df_union.collect())

# Ejecución lazy: las transformaciones no se ejecutan hasta una acción (collect, count, etc.)
from pyspark.sql.functions import upper
df_lazy = df.withColumn("nombre_mayus", upper(df.nombre))
# Nada se ejecuta hasta que se llama a una acción:
print(df_lazy.collect())

# filter, select, withColumn, distinct, union son transformaciones (crean nuevos DataFrames)
# collect, count, take, first son acciones (disparan la ejecución)

# count: acción que cuenta los elementos
print(df.count())

# take: acción que devuelve los primeros n elementos
print(df.take(2))

# first: acción que devuelve el primer elemento
print(df.first())

In [0]:
df.explain()